# Clase 17 — Árboles de Decisión: El Modelo que se Explica Solo

**Diplomado en Data Science Aplicada con Python** · Arca Continental Ecuador × UDLA

---

**Objetivos de hoy:**
1. Cerrar lo pendiente de clase 16: curvas ROC, PR y umbral óptimo.
2. Entender la intuición detrás de los árboles de decisión.
3. Entrenar, visualizar y podar un `DecisionTreeClassifier`.
4. Comparar árbol vs logística en el mismo dataset.
5. Crear una demo interactiva con Gradio.

**Dataset:** Stroke Prediction (mismo de clase 16).

In [ ]:
# Imports generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)

---

## 0. Cargar y preparar los datos (igual que clase 16)

In [ ]:
URL = ("https://raw.githubusercontent.com/cmosquerat/"
       "arca-diplomado/main/clase-16/stroke.csv")
df = pd.read_csv(URL)
print(df.shape)
df.head()

In [ ]:
# Preparar
df = df.drop(columns=["id"])
df["bmi"] = df["bmi"].fillna(df["bmi"].median())
df_enc = pd.get_dummies(df, drop_first=True, dtype=int)

X = df_enc.drop(columns=["stroke"])
y = df_enc["stroke"]

print(f"Features: {X.shape[1]}")
print(f"Positivos: {y.mean()*100:.2f}%")

In [ ]:
# Train/test split con stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"% positivos train: {y_train.mean()*100:.2f}%")
print(f"% positivos test:  {y_test.mean()*100:.2f}%")

In [ ]:
# Escalar para logística (el árbol NO necesita escalado)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
# Entrenar logística (la usaremos para comparar después)
modelo_lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
modelo_lr.fit(X_train_s, y_train)

print(f"Accuracy train: {modelo_lr.score(X_train_s, y_train):.3f}")
print(f"Accuracy test:  {modelo_lr.score(X_test_s, y_test):.3f}")

---

## 1. Cierre de Clase 16 — Curvas ROC, PR y Umbral Óptimo

En clase 16 vimos la teoría pero no alcanzamos a codificar. Hagámoslo ahora.

### 1.1 Probabilidades del modelo

La logística no devuelve "sí/no". Devuelve una **probabilidad**. El umbral (por defecto 0.5) la convierte en decisión.

In [ ]:
# Probabilidades de la clase positiva
y_proba_lr = modelo_lr.predict_proba(X_test_s)[:, 1]

print("Primeras 10 probabilidades:")
print(y_proba_lr[:10].round(3))

In [ ]:
# Distribución de probabilidades por clase real
fig = px.histogram(
    pd.DataFrame({"proba": y_proba_lr, "real": y_test.map({0: "Sano", 1: "ACV"})}),
    x="proba", color="real", nbins=40, barmode="overlay", opacity=0.7,
    title="Distribución de probabilidades por clase real",
    labels={"proba": "Probabilidad predicha de ACV", "count": "Pacientes"},
    color_discrete_map={"Sano": "#2563EB", "ACV": "#C82B40"})
fig.show()

**Interpretación:** los pacientes con ACV real (rojo) tienden a tener probabilidades más altas, pero hay mucho solapamiento. El umbral de 0.5 es una convención — podemos moverlo.

### 1.2 Construcción punto a punto de la curva ROC

Antes de usar `roc_curve()` de sklearn, hagámoslo **a mano** con 3 umbrales para entender exactamente qué se calcula.

In [ ]:
# Cálculo manual de 3 puntos en la curva ROC
print(f"{'Umbral':>8} {'VP':>5} {'FN':>5} {'FP':>5} {'VN':>5} {'TPR':>8} {'FPR':>8}")
print("-" * 52)

for u in [0.2, 0.5, 0.8]:
    y_pred_u = (y_proba_lr >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    tpr = tp / (tp + fn)
    fpr = fp / (fp + tn)
    print(f"{u:8.2f} {tp:5d} {fn:5d} {fp:5d} {tn:5d} {tpr:8.3f} {fpr:8.3f}")

Cada fila es **un punto** de la curva ROC. `roc_curve()` hace esto para **todos** los umbrales posibles.

### 1.3 Curva ROC completa y AUC

In [ ]:
# Calcular curva ROC con sklearn
fpr_lr, tpr_lr, umbrales_roc = roc_curve(y_test, y_proba_lr)
auc_lr = roc_auc_score(y_test, y_proba_lr)

# Graficar con Plotly
fig = px.area(x=fpr_lr, y=tpr_lr,
    title=f"Curva ROC — Logística (AUC = {auc_lr:.3f})",
    labels={"x": "FPR (1 - Especificidad)", "y": "TPR (Recall)"})
fig.add_shape(type="line", line=dict(dash="dash", color="gray"),
              x0=0, x1=1, y0=0, y1=1)
fig.update_layout(template="plotly_white")
fig.show()

print(f"AUC = {auc_lr:.3f}")

**Interpretación:**
- **AUC** = probabilidad de que el modelo asigne un score más alto a un enfermo que a un sano (elegidos al azar).
- AUC ≥ 0.8 se considera "bueno".
- La línea diagonal punteada es el **azar** (AUC = 0.5). Cuanto más arriba-izquierda esté la curva, mejor.

### 1.4 ¿Cómo se calcula el AUC? — Regla del trapecio

In [ ]:
# Calcular AUC a mano con la regla del trapecio y comparar con sklearn
from sklearn.metrics import auc as sk_auc

# Regla del trapecio: area = sum( (x[i+1]-x[i]) * (y[i]+y[i+1]) / 2 )
auc_manual = 0
for i in range(len(fpr_lr) - 1):
    dx = fpr_lr[i+1] - fpr_lr[i]
    y_avg = (tpr_lr[i] + tpr_lr[i+1]) / 2
    auc_manual += dx * y_avg

print(f"AUC manual (trapecio): {auc_manual:.6f}")
print(f"AUC sklearn:           {sk_auc(fpr_lr, tpr_lr):.6f}")
print(f"AUC roc_auc_score:     {auc_lr:.6f}")
print("\n¡Los tres dan lo mismo! sklearn usa la regla del trapecio internamente.")

### 1.5 Construcción punto a punto de la curva PR

In [ ]:
# Cálculo manual de 3 puntos en la curva PR
print(f"{'Umbral':>8} {'VP':>5} {'FN':>5} {'FP':>5} {'Precision':>12} {'Recall':>8}")
print("-" * 55)

for u in [0.2, 0.5, 0.8]:
    y_pred_u = (y_proba_lr >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    prec_val = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec_val = tp / (tp + fn)
    print(f"{u:8.2f} {tp:5d} {fn:5d} {fp:5d} {prec_val:12.3f} {rec_val:8.3f}")

**Diferencia con ROC:** el eje X sigue siendo Recall (= TPR), pero el eje Y es **Precision** en vez de FPR. La precisión *ignora los verdaderos negativos* → se concentra en la clase positiva.

### 1.6 Curva Precision-Recall completa y AP

In [ ]:
# Calcular curva PR
prec_lr, rec_lr, umbrales_pr = precision_recall_curve(y_test, y_proba_lr)
ap_lr = average_precision_score(y_test, y_proba_lr)

# Graficar
fig = px.area(x=rec_lr, y=prec_lr,
    title=f"Curva Precision-Recall — Logística (AP = {ap_lr:.3f})",
    labels={"x": "Recall", "y": "Precision"})
fig.add_hline(y=y_test.mean(), line_dash="dash",
              annotation_text=f"baseline ({y_test.mean():.2%})")
fig.update_layout(template="plotly_white")
fig.show()

print(f"AP = {ap_lr:.3f}  (baseline = {y_test.mean():.3f})")

**Interpretación:**
- **AP** (Average Precision) = área bajo la curva PR, calculada como suma de rectángulos (no trapecios).
- Un AP de 0.30 en datos con 5% de positivos = el modelo es ~**6x mejor que el azar**.
- En datos muy desbalanceados, un AP de 0.30 ya es *notable* (baseline es ~0.05).

### 1.7 ¿Cómo se calcula el AP? — Suma de rectángulos

In [ ]:
# Calcular AP a mano (rectángulos) y comparar con sklearn
ap_manual = 0
for i in range(1, len(rec_lr)):
    delta_recall = rec_lr[i] - rec_lr[i-1]
    ap_manual += delta_recall * prec_lr[i]

print(f"AP manual (rectángulos): {ap_manual:.6f}")
print(f"AP sklearn:              {ap_lr:.6f}")
print("\nNota: sklearn usa rectángulos (no trapecios) para AP,")
print("porque la curva PR es 'escalonada' y los trapecios la sobrestiman.")

### 1.8 Umbral óptimo por costo de negocio

El costo del error **no es simétrico**: un FN (no detectar ACV) cuesta una vida; un FP cuesta un chequeo adicional.

In [ ]:
# Barrido de umbrales: precision, recall y F1
umbrales = np.arange(0.05, 0.95, 0.02)
registros = []
for u in umbrales:
    y_pred_u = (y_proba_lr >= u).astype(int)
    registros.append({
        "umbral": u,
        "precision": precision_score(y_test, y_pred_u, zero_division=0),
        "recall":    recall_score(y_test, y_pred_u),
        "f1":        f1_score(y_test, y_pred_u, zero_division=0),
    })
df_u = pd.DataFrame(registros)

fig = px.line(df_u.melt(id_vars="umbral"),
              x="umbral", y="value", color="variable",
              title="Precision, Recall y F1 vs Umbral")
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Costo esperado por umbral
COSTO_FN = 10_000   # no detectar un ACV
COSTO_FP = 200      # falsa alarma

costos = []
for u in umbrales:
    y_pred_u = (y_proba_lr >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    costos.append({"umbral": u, "costo": fn * COSTO_FN + fp * COSTO_FP})

df_c = pd.DataFrame(costos)
u_opt = df_c.loc[df_c["costo"].idxmin(), "umbral"]

fig = px.line(df_c, x="umbral", y="costo",
              title=f"Costo esperado vs Umbral (óptimo: {u_opt:.2f})")
fig.add_vline(x=u_opt, line_dash="dash", line_color="green",
              annotation_text=f"óptimo = {u_opt:.2f}")
fig.update_layout(template="plotly_white")
fig.show()

print(f"Umbral óptimo por costo: {u_opt:.2f}")

**Interpretación:** cuando el FN cuesta 50x más que el FP, el umbral óptimo está **muy por debajo** de 0.5. Preferimos más falsas alarmas que perder un ACV real.

---

## 2. Árboles de Decisión — Intuición

Un árbol de decisión es un **diagrama de flujo**: en cada nodo hace una pregunta sobre una variable y, según la respuesta, va por una rama u otra. Al final (en las hojas) da su predicción.

**Ventaja clave:** puedes *leer* el árbol y explicar a un gerente o médico por qué decidió algo.

---

## 3. Tu primer árbol — sin límites

A diferencia de la logística, el árbol **no necesita escalado**. Trabaja directamente con los datos originales.

In [ ]:
# Árbol sin límites (peligro de overfitting)
arbol_sin_limite = DecisionTreeClassifier(random_state=42)
arbol_sin_limite.fit(X_train, y_train)

print(f"Accuracy train: {arbol_sin_limite.score(X_train, y_train):.3f}")
print(f"Accuracy test:  {arbol_sin_limite.score(X_test, y_test):.3f}")
print(f"Profundidad:    {arbol_sin_limite.get_depth()}")
print(f"Hojas:          {arbol_sin_limite.get_n_leaves()}")

**Train = 1.0** → el árbol memorizó TODOS los datos. Eso es **overfitting**. Necesitamos podar.

### Ejercicio 1: Observa el overfitting (5 min)

1. ¿Cuántas hojas tiene el árbol sin límites?
2. ¿Cuál es la brecha entre accuracy de train y test?
3. ¿Por qué un accuracy de train = 1.0 es **malo**?

In [ ]:
# Tu código aquí


---

## 4. Podar el árbol con `max_depth`

In [ ]:
# Árbol podado
arbol = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42)
arbol.fit(X_train, y_train)

print(f"Accuracy train: {arbol.score(X_train, y_train):.3f}")
print(f"Accuracy test:  {arbol.score(X_test, y_test):.3f}")
print(f"Profundidad:    {arbol.get_depth()}")
print(f"Hojas:          {arbol.get_n_leaves()}")

La brecha entre train y test se redujo. Menos memorización, más generalización.

### 4.1 Visualizar el árbol

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(arbol, ax=ax, filled=True, rounded=True,
          feature_names=X_train.columns,
          class_names=["No ACV", "ACV"],
          fontsize=7, impurity=True)
plt.title("Árbol de decisión (max_depth=5, balanced)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Cómo leer el árbol:**
- Cada nodo muestra: la variable y umbral de la pregunta, el Gini, cuántas muestras, la distribución de clases.
- `filled=True` colorea según la clase mayoritaria.
- Sigue cualquier camino desde la raíz hasta una hoja: esa es la **explicación** de la predicción.

---

## 5. Buscar la profundidad óptima

In [ ]:
# Barrido de profundidades
resultados = []
for depth in range(1, 21):
    t = DecisionTreeClassifier(max_depth=depth,
            class_weight="balanced", random_state=42)
    t.fit(X_train, y_train)
    resultados.append({
        "depth": depth,
        "train_acc": t.score(X_train, y_train),
        "test_acc":  t.score(X_test, y_test),
        "recall_test": recall_score(y_test, t.predict(X_test)),
    })

df_depth = pd.DataFrame(resultados)
df_depth

In [ ]:
# Graficar train vs test
fig = px.line(df_depth.melt(id_vars="depth",
              value_vars=["train_acc", "test_acc"]),
              x="depth", y="value", color="variable",
              title="Overfitting: Train vs Test Accuracy por profundidad",
              labels={"depth": "max_depth", "value": "Accuracy"})
fig.update_layout(template="plotly_white")
fig.show()

mejor = df_depth.loc[df_depth["test_acc"].idxmax()]
print(f"Mejor max_depth: {int(mejor['depth'])}")
print(f"  Train: {mejor['train_acc']:.3f}  Test: {mejor['test_acc']:.3f}")

### Ejercicio 2: Profundidad óptima (10 min)

1. ¿A partir de qué profundidad el accuracy de test empieza a **bajar**?
2. Repite el barrido usando `recall_score` en vez de accuracy. ¿Cambia el óptimo?
3. Entrena el árbol final con la mejor profundidad.

In [ ]:
# Tu código aquí


---

## 6. Feature Importance — ¿Qué variables mandan?

In [ ]:
# Importancias del árbol
importancias = pd.DataFrame({
    "feature": X_train.columns,
    "importancia": arbol.feature_importances_
}).sort_values("importancia", ascending=True).tail(10)

fig = px.bar(importancias, x="importancia", y="feature",
             orientation="h",
             title="Top 10 variables más importantes (Árbol)",
             color_discrete_sequence=["#C82B40"])
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# Comparar con coeficientes de logística
coefs_lr = pd.DataFrame({
    "feature": X_train.columns,
    "coef_abs": np.abs(modelo_lr.coef_[0])
}).sort_values("coef_abs", ascending=True).tail(10)

fig = px.bar(coefs_lr, x="coef_abs", y="feature",
             orientation="h",
             title="Top 10 variables más importantes (Logística — |coef|)",
             color_discrete_sequence=["#2563EB"])
fig.update_layout(template="plotly_white")
fig.show()

**Interpretación:** ambos modelos coinciden en que **edad** es la variable más importante. Pero difieren en las siguientes — eso es normal, miden importancia de formas distintas.

---

## 7. Comparación final: Árbol vs Logística

In [ ]:
# Predicciones y probabilidades de ambos
y_pred_arbol = arbol.predict(X_test)
y_prob_arbol = arbol.predict_proba(X_test)[:, 1]

y_pred_lr = modelo_lr.predict(X_test_s)
y_prob_lr = modelo_lr.predict_proba(X_test_s)[:, 1]

print("=== ÁRBOL ===")
print(classification_report(y_test, y_pred_arbol,
                            target_names=["No ACV", "ACV"]))
print(f"AUC:  {roc_auc_score(y_test, y_prob_arbol):.3f}")
print(f"AP:   {average_precision_score(y_test, y_prob_arbol):.3f}")

print("\n=== LOGÍSTICA ===")
print(classification_report(y_test, y_pred_lr,
                            target_names=["No ACV", "ACV"]))
print(f"AUC:  {roc_auc_score(y_test, y_prob_lr):.3f}")
print(f"AP:   {average_precision_score(y_test, y_prob_lr):.3f}")

### 7.1 Ambas curvas ROC en un solo gráfico

In [ ]:
fpr_t, tpr_t, _ = roc_curve(y_test, y_prob_arbol)
fpr_l, tpr_l, _ = roc_curve(y_test, y_prob_lr)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_l, y=tpr_l, mode="lines",
    name=f"Logística (AUC={roc_auc_score(y_test, y_prob_lr):.3f})",
    line=dict(color="#2563EB")))
fig.add_trace(go.Scatter(x=fpr_t, y=tpr_t, mode="lines",
    name=f"Árbol (AUC={roc_auc_score(y_test, y_prob_arbol):.3f})",
    line=dict(color="#C82B40")))
fig.add_shape(type="line", line=dict(dash="dash", color="gray"),
              x0=0, x1=1, y0=0, y1=1)
fig.update_layout(title="Curva ROC: Logística vs Árbol",
    xaxis_title="FPR", yaxis_title="TPR",
    template="plotly_white")
fig.show()

### 7.2 Ambas curvas PR en un solo gráfico

In [ ]:
prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_arbol)
prec_l, rec_l, _ = precision_recall_curve(y_test, y_prob_lr)

fig = go.Figure()
fig.add_trace(go.Scatter(x=rec_l, y=prec_l, mode="lines",
    name=f"Logística (AP={average_precision_score(y_test, y_prob_lr):.3f})",
    line=dict(color="#2563EB")))
fig.add_trace(go.Scatter(x=rec_t, y=prec_t, mode="lines",
    name=f"Árbol (AP={average_precision_score(y_test, y_prob_arbol):.3f})",
    line=dict(color="#C82B40")))
fig.add_hline(y=y_test.mean(), line_dash="dash",
              annotation_text=f"baseline ({y_test.mean():.2%})")
fig.update_layout(title="Curva PR: Logística vs Árbol",
    xaxis_title="Recall", yaxis_title="Precision",
    template="plotly_white")
fig.show()

### Ejercicio 3: Comparación completa (12 min)

1. ¿Cuál modelo tiene mejor AUC? ¿Y mejor AP?
2. ¿Cuál modelo elegirían para el hospital y **por qué**?
3. Si el gerente pide una explicación de por qué un paciente fue marcado como riesgo, ¿cuál modelo es más fácil de explicar?

In [ ]:
# Tu código aquí


---

### Ejercicio 4 (Retador): Árbol como reglas de negocio (10 min)

1. Entrena un árbol con `max_depth=3` (solo 3 preguntas).
2. Visualízalo con `plot_tree`.
3. Usa `export_text` para ver las reglas como texto.
4. Escribe en texto las "reglas" que el árbol aprendió.
5. ¿Estas reglas tienen sentido clínico? ¿Un médico las aceptaría?

In [ ]:
# Tu código aquí


---

## 8. Gradio — Del notebook a la demo interactiva

Ya saben entrenar, evaluar y comparar modelos. Pero un notebook no se le entrega a un médico ni a un gerente.

**La mayoría de APIs de predicción funcionan igual:** input → `modelo.predict()` → output. Ya sea Flask, FastAPI, Streamlit, Dash o Gradio — todos siguen ese patrón. **Gradio** es el más simple de todos y funciona dentro de Jupyter.

In [ ]:
# Instalar gradio si no lo tienen
# !pip install gradio

### 8.1 Crear la función de predicción

Necesitamos una función que reciba los datos del paciente y devuelva la predicción. El truco es construir el DataFrame con **exactamente las mismas columnas** que el modelo espera.

In [ ]:
import gradio as gr

# Guardar las columnas que el modelo espera
columnas_modelo = X_train.columns.tolist()

def predecir_acv(edad, glucosa, bmi, hipertension, enfermedad_cardiaca,
                 genero, casado, tipo_trabajo, residencia, tabaquismo):
    """Recibe datos de un paciente y devuelve el riesgo de ACV."""
    # Construir un dict con los datos crudos
    paciente = {
        "age": edad,
        "avg_glucose_level": glucosa,
        "bmi": bmi,
        "hypertension": int(hipertension),
        "heart_disease": int(enfermedad_cardiaca),
        "gender": genero,
        "ever_married": casado,
        "work_type": tipo_trabajo,
        "Residence_type": residencia,
        "smoking_status": tabaquismo,
    }
    
    # Crear DataFrame y aplicar el mismo encoding
    df_input = pd.DataFrame([paciente])
    df_input_enc = pd.get_dummies(df_input, drop_first=True, dtype=int)
    
    # Asegurar que tenga las mismas columnas que el train
    for col in columnas_modelo:
        if col not in df_input_enc.columns:
            df_input_enc[col] = 0
    df_input_enc = df_input_enc[columnas_modelo]
    
    # Predecir con el árbol (no necesita escalado)
    proba = arbol.predict_proba(df_input_enc)[0, 1]
    
    # Interpretar
    if proba >= 0.5:
        nivel = "ALTO"
    elif proba >= 0.2:
        nivel = "MEDIO"
    else:
        nivel = "BAJO"
    
    return f"Riesgo de ACV: {proba:.1%} ({nivel})"

### 8.2 Crear la interfaz

In [ ]:
demo = gr.Interface(
    fn=predecir_acv,
    inputs=[
        gr.Slider(0, 100, value=50, step=1, label="Edad"),
        gr.Slider(50, 300, value=100, step=1, label="Glucosa promedio (mg/dL)"),
        gr.Slider(10, 60, value=25, step=0.5, label="BMI"),
        gr.Checkbox(label="Hipertensión"),
        gr.Checkbox(label="Enfermedad cardíaca"),
        gr.Dropdown(["Male", "Female"], value="Male", label="Género"),
        gr.Dropdown(["Yes", "No"], value="Yes", label="Casado/a"),
        gr.Dropdown(["Private", "Self-employed", "Govt_job", "children", "Never_worked"],
                    value="Private", label="Tipo de trabajo"),
        gr.Dropdown(["Urban", "Rural"], value="Urban", label="Residencia"),
        gr.Dropdown(["never smoked", "formerly smoked", "smokes", "Unknown"],
                    value="never smoked", label="Tabaquismo"),
    ],
    outputs=gr.Text(label="Resultado"),
    title="Predictor de Riesgo de ACV",
    description="Ingresa los datos del paciente para obtener una estimación del riesgo de ACV.",
)

demo.launch()

### Ejercicio 5: Prueba tu demo (10 min)

1. Prueba con un paciente de **70 años**, glucosa 220, hipertenso, fumador.
2. Prueba con uno de **25 años**, glucosa 80, sano, nunca fumó.
3. ¿Los resultados tienen sentido clínico?
4. **Bonus:** cambia `arbol` por `modelo_lr` en la función (recuerda escalar con `scaler.transform`). ¿Cambian las predicciones?
5. **Bonus 2:** usa `demo.launch(share=True)` para obtener un link público. Compártelo con un compañero.

In [ ]:
# Tu código aquí


---

## Resumen

| Concepto | Lo que aprendimos |
|----------|-------------------|
| **Curva ROC + AUC** | Evalúa el modelo en todos los umbrales. AUC ≥ 0.8 = bueno. |
| **Curva PR + AP** | Mejor que ROC para datos desbalanceados. |
| **Umbral óptimo** | Se elige con lógica de negocio (costo FN vs FP). |
| **Árbol de decisión** | Diagrama de flujo: preguntas sí/no sobre las variables. |
| **Gini** | Mide impureza. El árbol elige el corte que más la reduce. |
| **Overfitting** | Sin podar, el árbol memoriza. Podar con `max_depth`. |
| **Feature importance** | Cuánto reduce la impureza cada variable. |
| **Gradio** | Del notebook a la demo interactiva en 5 líneas. Patrón: input → predict → output. |

**Próxima clase:** Random Forest — *¿qué pasa si entrenamos 100 árboles y los hacemos votar?*